Sam Voting Returns Cleanning Exploration

In [57]:
import pandas as pd
import numpy as np

file = "../data/raw/countypres_2000-2024.csv"
raw_df = pd.read_csv(file)
df = raw_df.copy()
#df.head()
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 94151 entries, 0 to 94150
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   state           94151 non-null  object 
 1   county_name     94151 non-null  object 
 2   year            94151 non-null  int64  
 3   state_po        94151 non-null  object 
 4   county_fips     94099 non-null  float64
 5   office          94151 non-null  object 
 6   candidate       94151 non-null  object 
 7   party           93650 non-null  object 
 8   candidatevotes  94114 non-null  float64
 9   totalvotes      94151 non-null  int64  
 10  version         94151 non-null  int64  
 11  mode            91356 non-null  object 
dtypes: float64(2), int64(3), object(7)
memory usage: 8.6+ MB


In [58]:
df = df.loc[
    (df["year"] >= 2016) # filter irrelevant years
    & (df["county_fips"].notna()) # remove fip nas
    & (df["mode"].fillna("").str.strip().str.lower() == "total") # dont break out by voting method
    & (pd.to_numeric(df["candidatevotes"], errors="coerce").fillna(0)) # remove zero votes and non-numeric values
].copy()

df.drop(columns=["version", "totalvotes", 'office', 'state_po', 'mode'], errors="ignore", inplace=True)
df["candidatevotes"] = pd.to_numeric(df["candidatevotes"], errors="coerce").astype("Int64") # coerce to numeric and set errors to nan
df["county_fips"] = pd.to_numeric(df["county_fips"], errors="coerce").astype("Int64")

df.head()

,state,county_name,year,county_fips,candidate,party,candidatevotes
0,ALABAMA,AUTAUGA,2024,1001,OTHER,OTHER,293
1,ALABAMA,AUTAUGA,2024,1001,CHASE OLIVER,LIBERTARIAN,65
2,ALABAMA,AUTAUGA,2024,1001,KAMALA D HARRIS,DEMOCRAT,7439
3,ALABAMA,AUTAUGA,2024,1001,DONALD J TRUMP,REPUBLICAN,20484
4,ALABAMA,BALDWIN,2024,1003,OTHER,OTHER,1276


In [59]:
file2 = "../data/processed/master_dataset_scaled.csv"
processed_df = pd.read_csv(file2)
df_vol = processed_df.copy()
#df.head()
df_vol.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 750 entries, 0 to 749
Data columns (total 39 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   county_fips                  750 non-null    int64  
 1   state                        750 non-null    object 
 2   county_name                  750 non-null    object 
 3   election_year                750 non-null    int64  
 4   median_age                   750 non-null    float64
 5   pct_black                    750 non-null    float64
 6   pct_asian                    750 non-null    float64
 7   pct_two_or_more_races        750 non-null    float64
 8   pct_hispanic                 750 non-null    float64
 9   pct_non_hispanic_white       750 non-null    float64
 10  pct_hs_or_higher             750 non-null    float64
 11  pct_bachelors_plus           750 non-null    float64
 12  pct_below_poverty            750 non-null    float64
 13  pct_income_under_25k

In [60]:
df_vol = df_vol[["county_fips", "state", "county_name", "election_year", "dem_votes", "rep_votes", "total_votes", "dem_pct", "rep_pct", "dem_margin"]]
#df.head()

df_piv = (
    df_vol
      .pivot(index=["county_fips", "county_name"], columns="election_year", values="dem_margin")
      .rename(columns=lambda y: f"dem_margin_{int(y)}")
      .reset_index()
)

df_piv

election_year,county_fips,county_name,dem_margin_2016,dem_margin_2020,dem_margin_2024
0,26001,Alcona County,-0.696285,-0.633409,-0.771109
1,26003,Alger County,0.102894,0.171466,0.127939
2,26005,Allegan County,-0.247255,-0.096421,-0.177021
3,26007,Alpena County,-0.236738,-0.193599,-0.235568
4,26009,Antrim County,-0.275470,-0.034639,-0.034924
...,...,...,...,...,...
245,42125,Washington County,-0.073925,0.005197,-0.106822
246,42127,Wayne County,-0.639480,-0.440976,-0.583382
247,42129,Westmoreland County,-0.342266,-0.224985,-0.226903
248,42131,Wyoming County,-0.630574,-0.506155,-0.572205


In [61]:
# get directional volatility index

df_piv["d_16_20"] = df_piv["dem_margin_2020"] - df_piv["dem_margin_2016"]
df_piv["d_20_24"] = df_piv["dem_margin_2024"] - df_piv["dem_margin_2020"]
df_piv["swing_dir_score"] = df_piv["d_16_20"] * df_piv["d_20_24"] * 10000

#df_piv.head(50)

In [66]:
# get general volatility index

import numpy as np

df_piv = df_piv.copy()

# abs swing each cycle
df_piv["swing_mag_16_20"] = df_piv["d_16_20"].abs()
df_piv["swing_mag_20_24"] = df_piv["d_20_24"].abs()

# z scores for magnitude of swing each cycle
mean_16_20 = df_piv["swing_mag_16_20"].mean()
sd_16_20 = df_piv["swing_mag_16_20"].std()

mean_20_24 = df_piv["swing_mag_20_24"].mean()
sd_20_24 = df_piv["swing_mag_20_24"].std()

# change vs average change by election
df_piv["z_swing_mag_16_20"] = (df_piv["swing_mag_16_20"] - mean_16_20) / sd_16_20
df_piv["z_swing_mag_20_24"] = (df_piv["swing_mag_20_24"] - mean_20_24) / sd_20_24

# change vs average change in total
df_piv["vol_z_abs_sum"] = df_piv["z_swing_mag_16_20"] + df_piv["z_swing_mag_20_24"]

# euclidean distance 
df_piv["vol_z_euc"] = np.sqrt(df_piv["z_swing_mag_16_20"]**2 + df_piv["z_swing_mag_20_24"]**2)

path = "../data/processed/county_volatility_dimTable.csv"
df_piv.to_csv(path, index=False)

In [63]:
file_nonscaled = "../data/processed/master_dataset.csv"
nonscaled_df = pd.read_csv(file_nonscaled)
df_ns = nonscaled_df.copy()

In [64]:
race_cols = df_ns[["pct_non_hispanic_white", "pct_black", "pct_asian"]]

# find other to get full distribution
race_cols["pct_other"] = (100 - race_cols["pct_non_hispanic_white"] - race_cols["pct_black"] - race_cols["pct_asian"])

# normalize to get proportions
race_cols = race_cols / 100.0

# pseudo zero for log
pseudo_zero = .00000001

# shannon entropy index
race_entropy = -(race_cols.clip(lower=pseudo_zero) * np.log(race_cols.clip(lower=pseudo_zero))).sum(axis=1)

# normalize from 0-1
race_entropy_norm = race_entropy / np.log(race_cols.shape[1])

df_ns["race_entropy"] = race_entropy
df_ns["race_entropy_norm"] = race_entropy_norm
 
# condense table for merge with scaled
entropy_cols = df_ns[["county_fips", "election_year","race_entropy_norm"]].copy()
df_ns.head()   

C:\Users\sgood\AppData\Local\Temp\ipykernel_20708\621169938.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  race_cols["pct_other"] = (100 - race_cols["pct_non_hispanic_white"] - race_cols["pct_black"] - race_cols["pct_asian"])


,county_fips,state,county_name,election_year,total_population,median_age,pct_black,pct_asian,pct_two_or_more_races,pct_hispanic,...,pct_young_adult_18_24,pct_foreign_born,dem_votes,rep_votes,total_votes,dem_pct,rep_pct,dem_margin,race_entropy,race_entropy_norm
0,26001,MI,Alcona County,2016,10461,57.4,0.248542,0.411051,1.309626,1.386101,...,2.619252,0.630915,1732.0,4201.0,6198.0,27.944498,67.779929,-39.835431,0.185556,0.133851
1,26003,MI,Alger County,2016,9396,49.0,7.322265,0.276713,4.129417,1.351639,...,5.119200,0.521499,1663.0,2585.0,4518.0,36.808322,57.215582,-20.407260,0.550228,0.396906
2,26005,MI,Allegan County,2016,113666,40.0,1.429627,0.653670,2.075379,7.081273,...,4.017032,1.454261,18050.0,34183.0,55786.0,32.355788,61.275230,-28.919442,0.414130,0.298732
3,26007,MI,Alpena County,2016,28929,47.3,0.594559,0.400982,1.541706,1.296277,...,3.840437,0.515054,4877.0,9090.0,14698.0,33.181385,61.845149,-28.663764,0.192863,0.139122
4,26009,MI,Antrim County,2016,23215,50.0,0.340297,0.267069,1.791945,1.955632,...,3.372819,0.960586,4448.0,8469.0,13582.0,32.749227,62.354587,-29.605360,0.215606,0.155527


In [65]:
file_scaled = "../data/processed/master_dataset_scaled.csv"
master_scaled = pd.read_csv(file_scaled)

master_scaled = master_scaled.merge(
    entropy_cols,
    on=["county_fips", "election_year"],
    how="left",
    validate="one_to_one"
)

#master_scaled.head()

path = "../data/processed/master_dataset_scaled.csv"
master_scaled.to_csv(path, index=False)